# 5장 배포 판단 검토

## 이번 질문

데이터, 모델, 서빙, 운영 근거를 하나의 배포 판단으로 연결합니다. Candidate A는 배포 경로에서 제외되고 Candidate B만 승인 오버레이에 포함되는지 확인합니다. Candidate B의 모델 승인은 `APPROVE`로 유지하며, 대상 API와 운영 기록을 실제로 보지 못한 경우 운영 환경 확인 상태는 `target_pending`으로 기록합니다.

## 먼저 예상

Candidate B의 모델 승인이 `APPROVE`여도 대상 API와 운영 근거가 없을 때 어떤 운영 환경 확인 상태를 골라야 할지 먼저 적습니다. 어떤 조건이면 되돌리기 검토를 열지도 예상합니다.

## 실행과 관측

In [ ]:
from pathlib import Path

import json
import pandas as pd

# 1. 저장소 루트를 찾는다.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# 2. 공식 평가 JSON에서 후보 판단만 먼저 본다.
canonical_path = ROOT / "docs/evidence/model-v2/canonical-benchmark.json"
canonical = json.loads(canonical_path.read_text(encoding="utf-8"))
decisions = pd.DataFrame(canonical["decisions"]).set_index("profile")
decisions[["decision", "checks"]]


### 1. 공식 평가 뒤의 배포 선언

배포 선언이 방금 본 공식 평가 파일 지문을 그대로 가리키는지 확인한다.


In [ ]:
# 1. 배포 선언이 공식 평가 파일 지문을 가리키는지 확인한다.
import hashlib

release_manifest_path = ROOT / "docs/evidence/model-v2/release-manifest.json"
final_benchmark_path = ROOT / "docs/evidence/model-v2/final-benchmark.json"
release_manifest = json.loads(release_manifest_path.read_text(encoding="utf-8"))

canonical_digest = hashlib.sha256(canonical_path.read_bytes()).hexdigest()
final_digest = hashlib.sha256(final_benchmark_path.read_bytes()).hexdigest()

release_chain = {
    "release_status": release_manifest["release_status"],
    "approved_profile": release_manifest["approved_profile"],
    "canonical_digest_matches": release_manifest["canonical_evidence"]["sha256"]
    == canonical_digest,
    "final_digest_matches": release_manifest["final_evidence"]["sha256"]
    == final_digest,
    "freeze_digest_matches": release_manifest["freeze_manifest"]["sha256"]
    == canonical["sealed_test"]["freeze_manifest_sha256"],
    "historical_reconciliation": release_manifest["historical_reconciliation"],
}
assert release_chain["release_status"] == "release_approved"
assert release_chain["approved_profile"] == "candidate-b"
assert release_chain["canonical_digest_matches"]
assert release_chain["final_digest_matches"]
assert release_chain["freeze_digest_matches"]

pd.DataFrame({"값": list(release_chain.values())}, index=list(release_chain.keys()))


### 2. 배포 오버레이와 승인 결과 연결

세 overlay 문자열에 Candidate A가 없고, B overlay만 Candidate B 지문을 쓰는지 본다.


In [ ]:
import yaml

overlay_root = ROOT / "deploy/k8s"

# 1. overlay 폴더의 YAML을 문자열로 이어 읽는다.
overlays = {}
for name in ("baseline", "candidate-b", "rollback"):
    parts = []
    for path in sorted((overlay_root / name).glob("*.yaml")):
        parts.append(path.read_text(encoding="utf-8"))
    overlays[name] = "\n".join(parts)

# 2. 기본 모델 지문은 env 파일에 있다.
base_identity = {}
for line in (
    ROOT / "deploy/k8s/base/config/model-identity.env"
).read_text(encoding="utf-8").splitlines():
    if not line or line.startswith("#"):
        continue
    key, value = line.split("=", maxsplit=1)
    base_identity[key] = value

# 3. candidate-b overlay가 덮어쓰는 지문.
candidate_kustomization = yaml.safe_load(
    (overlay_root / "candidate-b/kustomization.yaml").read_text(encoding="utf-8")
)
candidate_digest = None
for item in candidate_kustomization["configMapGenerator"]:
    if item["name"] == "model-identity":
        candidate_digest = item["literals"][0].split("=", maxsplit=1)[1]
        break

expected_digests = {
    "baseline": base_identity["AIQA_KSERVE_EXPECTED_MODEL_SHA256"],
    "candidate-b": candidate_digest,
    "rollback": base_identity["AIQA_KSERVE_EXPECTED_MODEL_SHA256"],
}
profiles = {
    "baseline": "baseline",
    "candidate-b": "candidate-b",
    "rollback": "baseline",
}

# 4. 각 overlay 텍스트에 어떤 모델 이름이 있는지 표시한다.
rows = []
for name, document in overlays.items():
    rows.append(
        {
            "overlay": name,
            "contains_baseline": "baseline-f2576f12512a" in document,
            "contains_candidate_b": "candidate-b-c712a8e52344" in document,
            "contains_candidate_a": "candidate-a" in document,
            "expected_model_sha256": expected_digests[name],
            "manifest_model_sha256": release_manifest["model_bundles"][
                f"{profiles[name]}/model.joblib"
            ],
        }
    )
summary = pd.DataFrame(rows).set_index("overlay")
summary


### 3. 판단 기록 구성

모델 승인 칸과 운영 확인 칸을 따로 적는다. 대상 API를 아직 안 봤으면 운영 칸은 target_pending 이다.


In [ ]:
# 1. 모델 승인 칸과 운영 확인 칸을 따로 적는다.
# 모델 승인: 공식 평가 JSON의 HOLD / APPROVE.
# 운영 범위: 대상 API와 운영 기록을 아직 보지 못했으므로 target_pending.
release_record = {
    "dataset_sha256": canonical["sealed_test"]["dataset_sha256"],
    "sealed_test_status": canonical["sealed_test"]["status"],
    "candidate_a": decisions.loc["candidate-a", "decision"],
    "candidate_b": decisions.loc["candidate-b", "decision"],
    "deployment_allowed": canonical["deployment_allowed"],
    "model_approval": {
        "candidate-a": decisions.loc["candidate-a", "decision"],
        "candidate-b": decisions.loc["candidate-b", "decision"],
    },
    "operational_deployment_scope": "target_pending",
    "current_recommendation": "target evidence collection",
    "false_approval_risk": "unobserved target bundle or telemetry condition",
    "false_hold_risk": "delaying a frozen-approved Candidate B without model evidence",
    "reassessment_on_identity_mismatch": "rollback_required",
    "approved_overlay": "candidate-b",
    "rollback_overlay": "rollback",
    "owner_next_action": "Serving/Platform collects target identity and telemetry.",
}
pd.DataFrame(
    {
        "값": [
            release_record["candidate_a"],
            release_record["candidate_b"],
            release_record["operational_deployment_scope"],
            release_record["current_recommendation"],
        ]
    },
    index=["candidate_a", "candidate_b", "operational_scope", "current_recommendation"],
)


### 4. 대상 환경의 모델 정보 확인

URL이 있을 때만 `/v1/model`을 GET한다. 프로필이 달라도 운영 기록이 없으면 대상 확인을 완료라고 쓰지 않는다.


In [ ]:
# 1. URL이 있을 때만 대상 API의 모델 정보를 읽는다.
import os

import requests

API_URL = os.getenv("AIQA_RISK_API_URL")
expected_profile = os.getenv("AIQA_EXPECTED_PROFILE")

if API_URL is None:
    deployed_identity = {
        "status": "URL_NOT_CONFIGURED",
        "next_action": "Use the instructor-provided Risk API URL after GitOps sync.",
    }
else:
    try:
        response = requests.get(f"{API_URL.rstrip('/')}/v1/model", timeout=3)
        response.raise_for_status()
        deployed_identity = response.json()
        observed_profile = deployed_identity.get("profile")
        deployed_identity["expected_profile"] = expected_profile
        deployed_identity["profile_matches"] = (
            expected_profile is None or observed_profile == expected_profile
        )
    except requests.RequestException as error:
        deployed_identity = {"status": "API_UNREACHABLE", "detail": str(error)}

if deployed_identity.get("profile_matches") is False:
    release_record["operational_deployment_scope"] = "rollback_required"
    release_record["current_recommendation"] = "rollback review"
    release_record["operational_reason"] = (
        "observed API profile differs from expected profile"
    )
else:
    release_record["operational_reason"] = (
        "API identity alone is insufficient; target telemetry is still required."
    )

pd.DataFrame(
    {
        "값": [
            deployed_identity.get("status") or deployed_identity.get("profile"),
            release_record["operational_deployment_scope"],
            release_record["current_recommendation"],
        ]
    },
    index=["identity", "operational_scope", "current_recommendation"],
)


## 해석과 기록

모델 승인과 운영 환경 확인 상태를 별도 항목으로 쓰고, 현재 권고에는 확인하지 못한 대상 근거와 다음 담당자를 함께 남깁니다.

## 결과 점검

In [ ]:
assert release_record["candidate_a"] == "HOLD"
assert release_record["candidate_b"] == "APPROVE"
assert release_record["deployment_allowed"] is True
assert release_record["model_approval"]["candidate-b"] == "APPROVE"
assert release_record["operational_deployment_scope"] in {
    "target_pending",
    "rollback_required",
}
assert summary.loc["candidate-b", "contains_candidate_b"]
assert not summary["contains_candidate_a"].any()
assert summary.loc["rollback", "contains_baseline"]
assert (summary["expected_model_sha256"] == summary["manifest_model_sha256"]).all()
print("Release decision checks passed.")


## 다음 확인

3장에서 공유 Argo의 `tta` 계정으로 Candidate B overlay를 동기화한 뒤 예상 모델 프로필, 변경되지 않는 모델 식별 정보, Grafana 운영 기록의 시간 범위를 같은 배포 판단 기록에 남깁니다. 모델 프로필 불일치, 정상 요청의 입력 규약 실패, 운영 환경 담당자가 확인한 이상 상태가 있으면 `rollback_required` 검토를 열 수 있습니다. 대상 환경 정보나 Grafana 접속 정보가 없으면 배포 또는 되돌리기의 성공을 주장하지 않고 `target_pending`과 다음 담당자의 조치를 유지합니다.